# Lecture 13 — GPU Acceleration

**CMU 10-414/714, Deep Learning Systems (lecture by Tianqi Chen)**
Companion notebook · Sangram Lembe

There is no GPU in the environment this notebook was built in, and CUDA cannot be run from plain
Python. So instead of timing a GPU, this notebook **simulates the CUDA execution model** — grids,
blocks, threads, global memory, shared memory, and `__syncthreads()` — and counts every memory read
exactly. That turns the lecture's claims into numbers you can check:

1. how a thread finds its global index, and why the grid needs a guard
2. which loops can be handed one-element-per-thread, and a clever way around the ones that cannot
3. shared memory cutting global reads from 20 to 8 in the window-sum example
4. what goes wrong **without** `__syncthreads()`
5. two-level tiling for matrix multiply, and the occupancy trade-off

In [1]:
import numpy as np
rng = np.random.default_rng(0)

class GlobalMemory:
    """An array that counts how many times it is read - GPU global memory."""
    def __init__(self, data):
        self.data, self.reads = np.array(data, dtype=float), 0
    def __getitem__(self, i):
        self.reads += 1
        return self.data[i]
    def __setitem__(self, i, v):
        self.data[i] = v
    def __len__(self):
        return len(self.data)

def launch(kernel, grid_dim, block_dim, *args):
    """Run kernel once for every (blockIdx, threadIdx) - what <<<grid, block>>> does."""
    for block_idx in range(grid_dim):
        for thread_idx in range(block_dim):
            kernel(block_idx, thread_idx, block_dim, *args)

print("simulator ready")

simulator ready


## 1. One program, many threads

You write the code for **one** thread. Every thread runs it; the only difference is its position.
The CUDA vector-add kernel, line for line:

```c
__global__ void VecAdd(float* A, float* B, float* C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) C[i] = A[i] + B[i];
}
```

In [2]:
def vec_add(blockIdx, threadIdx, blockDim, A, B, C, n):
    i = blockIdx * blockDim + threadIdx          # my global position
    if i < n:                                    # the guard
        C[i] = A[i] + B[i]

# the lecture's picture: 2 blocks of 4 threads
print(f"{'blockIdx':>9}{'threadIdx':>11}{'global i':>10}")
for b in range(2):
    for t in range(4):
        print(f"{b:>9}{t:>11}{b*4 + t:>10}")

 blockIdx  threadIdx  global i
        0          0         0
        0          1         1
        0          2         2
        0          3         3
        1          0         4
        1          1         5
        1          2         6
        1          3         7


In [3]:
n, threads_per_block = 1000, 512
blocks = (n + threads_per_block - 1) // threads_per_block      # round UP
A, B = rng.random(n), rng.random(n)
C = np.zeros(n)
launch(vec_add, blocks, threads_per_block, A, B, C, n)

print(f"n = {n}, {threads_per_block} threads per block -> {blocks} blocks = {blocks*threads_per_block} threads")
print(f"spare threads stopped by the guard: {blocks*threads_per_block - n}")
print("result correct:", np.allclose(C, A + B))

def vec_add_no_guard(blockIdx, threadIdx, blockDim, A, B, C, n):
    i = blockIdx * blockDim + threadIdx
    C[i] = A[i] + B[i]
try:
    launch(vec_add_no_guard, blocks, threads_per_block, A, B, np.zeros(n), n)
except IndexError as e:
    print("without the guard:", type(e).__name__, "-", e)

n = 1000, 512 threads per block -> 2 blocks = 1024 threads
spare threads stopped by the guard: 24
result correct: True
without the guard: IndexError - index 1000 is out of bounds for axis 0 with size 1000


Rounding the block count up guarantees every element gets a thread, which means the last block has
spare threads. Without the `if (i < n)` guard they read and write past the end of the arrays — here an
`IndexError`; on a real GPU, silent memory corruption or a crash.

## 2. What can be parallelised

A GPU runs threads in no guaranteed order. Vector addition does not care, because every element is
independent. The lecture's counter-example is a loop where each step needs the previous one:

$$c_i = \max(c_{i-1}, 0) + a_i$$

Simulate "one thread per element, in whatever order the hardware picks":

In [4]:
a = rng.normal(size=16)

def sequential(a):
    c = np.empty_like(a)
    for i in range(len(a)):
        c[i] = (max(c[i-1], 0) if i else 0) + a[i]
    return c

def one_thread_per_element(a, order):
    c = np.zeros_like(a)                           # memory starts as zeros
    for i in order:                                # threads run in this order
        c[i] = (max(c[i-1], 0) if i else 0) + a[i]
    return c

truth = sequential(a)
for trial in range(3):
    order = rng.permutation(len(a))
    ok = np.allclose(one_thread_per_element(a, order), truth)
    print(f"random thread order {trial+1}: correct = {ok}")

order = rng.permutation(len(a))
va = rng.random(16); vb = rng.random(16); vc = np.zeros(16)
for i in order: vc[i] = va[i] + vb[i]
print("vector add in a random order: correct =", np.allclose(vc, va + vb))

random thread order 1: correct = False
random thread order 2: correct = False
random thread order 3: correct = False
vector add in a random order: correct = True


The recurrence breaks because a thread may read $c_{i-1}$ before its neighbour has written it.

### It *can* be parallelised — with a different algorithm

The lecture mentions parallel **scan**. Here it is for this exact recurrence. Each step is a function
$f_i(x) = \max(x, 0) + a_i = \max(x + a_i,\; a_i)$, and every function of the form
$\max(x + p, q)$ composes into another of the same form:

$$\max\big(\max(x + p_1, q_1) + p_2,\; q_2\big) = \max\big(x + (p_1 + p_2),\; \max(q_1 + p_2, q_2)\big)$$

Because composition is associative, we can combine neighbours in pairs, then pairs of pairs —
$\log_2 n$ rounds, where every combination inside a round is independent and could be one thread:

In [5]:
def parallel_scan(a):
    """Hillis-Steele scan over functions max(x + p, q). Returns c and the number of rounds."""
    P, Q = a.copy(), a.copy()                       # f_i(x) = max(x + a_i, a_i)
    step, rounds = 1, 0
    while step < len(a):
        P_new, Q_new = P.copy(), Q.copy()
        for i in range(step, len(a)):               # all independent -> one thread each
            p1, q1, p2, q2 = P[i-step], Q[i-step], P[i], Q[i]
            P_new[i] = p1 + p2
            Q_new[i] = max(q1 + p2, q2)
        P, Q = P_new, Q_new
        step, rounds = step * 2, rounds + 1
    return Q, rounds                                # applied to the empty start

par, rounds = parallel_scan(a)
print("parallel scan equals the sequential loop:", np.allclose(par, truth))
print(f"{len(a)} elements, {rounds} parallel rounds instead of {len(a)} dependent steps")

parallel scan equals the sequential loop: True
16 elements, 4 parallel rounds instead of 16 dependent steps


The deciding question is never "is there a loop" but **how independent the pieces of work are** — and
sometimes the dependence can be restructured away.

## 3. Shared memory: the sliding-window sum

Each output is the sum of 5 neighbouring inputs. The obvious kernel has every thread read its own 5
values from global memory.

In [6]:
W = 5

def window_naive(blockIdx, threadIdx, blockDim, inp, out):
    i = blockIdx * blockDim + threadIdx
    if i < len(out):
        out[i] = sum(inp[i + k] for k in range(W))          # 5 global reads per thread

n_out, TPB = 64, 4
inp = GlobalMemory(rng.random(n_out + W - 1))
out = np.zeros(n_out)
launch(window_naive, n_out // TPB, TPB, inp, out)
naive_reads = inp.reads
expected = np.array([inp.data[i:i+W].sum() for i in range(n_out)])
print("correct:", np.allclose(out, expected))
print(f"global reads: {naive_reads}  ({naive_reads // (n_out // TPB)} per block of {TPB} threads)")

correct: True
global reads: 320  (20 per block of 4 threads)


Four threads need only $4 + 5 - 1 = 8$ distinct values between them, but read them 20 times. With
shared memory, the block loads those 8 **once**, cooperatively, waits for everyone at a barrier, then
reads from the fast copy.

To simulate a barrier, each thread is written as a Python generator: `yield` marks `__syncthreads()`.
The scheduler runs every thread of a block up to the barrier, then every thread past it.

In [7]:
def window_shared(tid, base, inp, out, shared):
    TILE = 4
    shared[tid] = inp[base + tid]                    # cooperative load: 2 values each
    if tid < W - 1:
        shared[tid + TILE] = inp[base + tid + TILE]
    yield                                            # __syncthreads()
    out[base + tid] = sum(shared[tid + k] for k in range(W))   # fast reads

def run_block(kernel, TPB, base, inp, out, barrier=True):
    shared = np.zeros(TPB + W - 1)                   # this block's shared memory
    threads = [kernel(t, base, inp, out, shared) for t in range(TPB)]
    if barrier:
        for th in threads: next(th, None)            # everyone reaches the barrier...
        for th in threads: next(th, None)            # ...then everyone continues
    else:
        for th in threads:                           # each thread runs start to finish
            next(th, None); next(th, None)

inp = GlobalMemory(inp.data)
out = np.zeros(n_out)
for blk in range(n_out // TPB):
    run_block(window_shared, TPB, blk * TPB, inp, out)
print("correct:", np.allclose(out, expected))
print(f"global reads: {inp.reads}  ({inp.reads // (n_out // TPB)} per block)  vs {naive_reads} naive"
      f"  -> {naive_reads / inp.reads:.1f}x fewer")

correct: True
global reads: 128  (8 per block)  vs 320 naive  -> 2.5x fewer


From 20 global reads per block to 8, exactly as in the lecture.

### Why `__syncthreads()` matters

Now run the same kernel **without** the barrier, so each thread goes from start to finish before the
next one begins. Thread 0 starts summing before threads 1–3 have loaded their values into shared
memory:

In [8]:
inp = GlobalMemory(inp.data)
out_race = np.zeros(n_out)
for blk in range(n_out // TPB):
    run_block(window_shared, TPB, blk * TPB, inp, out_race, barrier=False)

wrong = ~np.isclose(out_race, expected)
print(f"outputs wrong without the barrier: {wrong.sum()} of {n_out}")
print("first block, correct :", np.round(expected[:4], 3))
print("first block, no sync :", np.round(out_race[:4], 3))

outputs wrong without the barrier: 48 of 64
first block, correct : [1.931 1.717 1.287 1.814]
first block, no sync : [1.183 1.534 1.235 1.814]


The early threads summed slots that were still zero. On real hardware the order is unpredictable, so
this kind of bug appears only sometimes — which is what makes missing barriers so painful to debug.

## 4. Matrix multiplication: two levels of tiling

Each **thread** computes a $V\times V$ tile of $C$ in registers; each **block** computes an $L\times L$
tile using $(L/V)^2$ threads, first loading the rows of $A$ and columns of $B$ it needs into shared
memory. Count the reads at both levels:

In [9]:
def gpu_matmul(A, B, L, V):
    """C = A @ B with block tiles L x L (shared) and thread tiles V x V (registers).
    Returns C, global->shared reads, shared->register reads."""
    n = A.shape[0]
    C = np.zeros((n, n))
    g_reads = s_reads = 0
    for bi in range(0, n, L):                         # one thread block per L x L tile
        for bj in range(0, n, L):
            sA = A[bi:bi+L, :]                        # cooperative load into shared memory
            sB = B[:, bj:bj+L]
            g_reads += sA.size + sB.size
            for ti in range(0, L, V):                 # (L/V)^2 threads in this block
                for tj in range(0, L, V):
                    c = np.zeros((V, V))              # registers
                    for k in range(n):
                        a = sA[ti:ti+V, k]            # V shared reads, each reused V times
                        b = sB[k, tj:tj+V]
                        s_reads += 2 * V
                        c += np.outer(a, b)
                    C[bi+ti:bi+ti+V, bj+tj:bj+tj+V] = c
    return C, g_reads, s_reads

n = 64
A, B = rng.random((n, n)), rng.random((n, n))
print(f"naive: every multiply reads 2 numbers from global memory = {2*n**3:,} reads\n")
print(f"{'L':>4}{'V':>4}{'threads/block':>15}{'global reads':>15}{'= 2n^3/L':>11}{'shared reads':>15}{'= 2n^3/V':>11}")
for L, V in [(8, 2), (16, 4), (32, 4), (32, 8)]:
    C, g, s = gpu_matmul(A, B, L, V)
    assert np.allclose(C, A @ B)
    print(f"{L:>4}{V:>4}{(L//V)**2:>15}{g:>15,}{2*n**3//L:>11,}{s:>15,}{2*n**3//V:>11,}")
print("\nevery configuration computes A @ B correctly")

naive: every multiply reads 2 numbers from global memory = 524,288 reads

   L   V  threads/block   global reads   = 2n^3/L   shared reads   = 2n^3/V


   8   2             16         65,536     65,536        262,144    262,144


  16   4             16         32,768     32,768        131,072    131,072
  32   4             64         16,384     16,384        131,072    131,072
  32   8             16         16,384     16,384         65,536     65,536

every configuration computes A @ B correctly


The counts match the lecture's reasoning: global traffic falls by a factor of $L$, and shared-memory
traffic by a factor of $V$. So why not make both huge?

## 5. The occupancy trade-off

A multiprocessor has a fixed pool of registers. Recent NVIDIA GPUs have 65,536 registers and at most
2,048 resident threads per multiprocessor — typical figures, used here for illustration. More
registers per thread (a bigger $V$) means fewer threads fit.

In [10]:
REGS_PER_SM, MAX_THREADS = 65_536, 2_048
print(f"{'registers/thread':>17}{'threads that fit':>18}{'share of maximum':>18}")
for r in (32, 64, 128, 255):
    fit = min(MAX_THREADS, REGS_PER_SM // r)
    print(f"{r:>17}{fit:>18,}{fit/MAX_THREADS:>18.0%}")

 registers/thread  threads that fit  share of maximum
               32             2,048              100%
               64             1,024               50%
              128               512               25%
              255               257               13%


Fewer resident threads matters because a GPU hides slow memory by **switching** to another thread
while one waits. A simple model: if each thread computes for $c$ cycles and then waits $w$ cycles for
memory, the core stays busy only if at least $(c + w)/c$ threads take turns.

In [11]:
c = 10                                          # cycles of compute between loads
for w in (100, 400):
    for threads in (4, 16, 64):
        busy = min(1.0, threads * c / (c + w))
        print(f"wait {w:>3} cycles, {threads:>3} threads taking turns -> cores busy {busy:5.0%}")
    print()

wait 100 cycles,   4 threads taking turns -> cores busy   36%
wait 100 cycles,  16 threads taking turns -> cores busy  100%
wait 100 cycles,  64 threads taking turns -> cores busy  100%

wait 400 cycles,   4 threads taking turns -> cores busy   10%
wait 400 cycles,  16 threads taking turns -> cores busy   39%
wait 400 cycles,  64 threads taking turns -> cores busy  100%



With too few threads, cores sit idle waiting for memory. That is the tension: large tiles reduce
memory traffic but leave fewer threads to hide the memory that remains. Since the best balance
depends on the exact GPU and problem size, practitioners **auto-tune** — try many values of $L$ and
$V$, measure, keep the fastest.

## Summary

- A thread's global index is `blockIdx * blockDim + threadIdx`; round the grid up and guard with
  `i < n` — without it, the simulator raised an out-of-bounds error.
- Independent work maps directly to threads. The dependent recurrence $c_i=\max(c_{i-1},0)+a_i$ gave
  wrong answers in random thread order — but a scan over composed functions computed it in 4 rounds
  instead of 16 steps.
- Shared memory cut the window sum from 20 global reads per block to 8.
- Without `__syncthreads()`, threads summed values that had not been loaded yet.
- Two-level tiling cuts global reads by $L$ and shared reads by $V$; bigger tiles mean fewer resident
  threads to hide memory latency, so tile sizes are tuned by measurement.